# DIMBA — Shakespeare (Kaggle GPU → HuggingFace)

**~9.7M params | fp16 mixed | ~10-15 min P100 | Auto-upload private to HF**

⚡ Kaggle → Settings → Secrets → Add `HF_TOKEN`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os, sys, torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}')

## 1. Setup

In [ ]:
!git clone https://github.com/devnull37/dimba-lib-exp.git /kaggle/working/dimba-lib-exp
import os, urllib.request
os.chdir('/kaggle/working/dimba-lib-exp')
!git checkout main && !pip install -q -e . && !pip install -q tensorboard huggingface_hub
os.makedirs('data', exist_ok=True)
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'data/shakespeare.txt')
print(f'Repo: {os.getcwd()}  |  Data: {os.path.getsize("data/shakespeare.txt"):,} bytes  |  Ready.')

## 2. Train

d_model=320, 5 layers, latent diffusion, fp16 — **~9.7M params, ~10-15 min**

In [ ]:
import os
os.chdir('/kaggle/working/dimba-lib-exp')
!python scripts/train_shakespeare.py \
    --d-model 320 --num-layers 5 \
    --seq-len 128 --num-steps 128 \
    --epochs 15 --batch-size 64 \
    --accelerator gpu --precision 16-mixed

## 3. Generate Samples

In [ ]:
import os, sys, torch
os.chdir('/kaggle/working/dimba-lib-exp')
sys.path.insert(0, 'src')
from dimba import DIMBA
from dimba.tokenizers import SimpleCharacterTokenizer
from dimba.diffusion.sampling import sample_from_model

ckpt = torch.load('checkpoints/shakespeare/shakespeare1.ckpt', map_location='cuda')
s = ckpt['state_dict']
num_layers = sum(1 for k in s if '.mamba_fwd.in_proj.weight' in k)
d_state   = s['model.denoiser.blocks.0.mamba_fwd.A_log'].shape[1]
d_latent  = s['model.denoiser.blocks.0.mamba_fwd.out_proj.weight'].shape[0]
vocab_sz  = s['model.token_embed.embedding.weight'].shape[0]
outer_d   = s['model.token_embed.embedding.weight'].shape[1]
d_prompt  = s['model.null_cond'].shape[0]
T         = s['model.noise_schedule.alphas_cumprod'].shape[0]
n_params  = sum(v.numel() for v in s.values()) // 2

model = DIMBA(
    vocab_size=vocab_sz, d_model=outer_d, d_prompt=d_prompt, d_latent=d_latent,
    num_denoiser_layers=num_layers, d_state=d_state, num_diffusion_steps=T,
    d_conv=4, expand=2, conditioning_type='film', dropout=0.1,
    use_weight_tying=True, use_simple_mamba=False,
    latent_diffusion=True,
).cuda()
model_state = {k[6:]: v for k, v in s.items() if k.startswith('model.')}
model.load_state_dict(model_state, strict=False)
model.eval()
print(f'{n_params:,} params | d_model={outer_d} latent={d_latent} layers={num_layers}')

tok = SimpleCharacterTokenizer(vocab_size=128)
for p in ['Juliet:', 'Hamlet:', 'To be or not to', 'Now is the winter of our', 'First Citizen:']:
    ids = torch.tensor([tok.encode(p)], device='cuda')
    with torch.no_grad():
        gen = sample_from_model(model, prompt_ids=ids, seq_len=200, num_steps=50, temperature=0.8, top_p=0.95)
    print(f'\n  {p}')
    print(f'  {tok.decode(gen[0])[:200]}')
    print('  ' + '-' * 50)

## 4. Upload to HuggingFace (private)

In [ ]:
import os
from huggingface_hub import HfApi, create_repo
from datetime import datetime

token = os.environ.get('HF_TOKEN')
if not token:
    print('ERROR: HF_TOKEN not set. Kaggle → Settings → Secrets → Add HF_TOKEN')
else:
    api = HfApi(token=token)
    username = api.whoami()['name']
    repo_id = f'{username}/dimba-shakespeare'
    create_repo(repo_id, private=True, token=token, exist_ok=True)
    
    os.chdir('/kaggle/working/dimba-lib-exp')
    api.upload_file(
        path_or_fileobj='checkpoints/shakespeare/shakespeare1.ckpt',
        path_in_repo='shakespeare1.ckpt',
        repo_id=repo_id, token=token)
    api.upload_file(
        path_or_fileobj='checkpoints/shakespeare/tokenizer.json',
        path_in_repo='tokenizer.json',
        repo_id=repo_id, token=token)
    
    card = f'''---
license: mit
tags: [dimba, diffusion, mamba, shakespeare]
---
# DIMBA Shakespeare
Latent-diffusion Mamba trained on tinyshakespeare.
- Params: {n_params:,}
- Config: d_model={outer_d}, latent={d_latent}, layers={num_layers}
- Trained: {datetime.now().strftime('%Y-%m-%d')}
'''
    api.upload_file(
        path_or_fileobj=card.encode(),
        path_in_repo='README.md',
        repo_id=repo_id, token=token)
    
    print(f'https://huggingface.co/{repo_id} (private)')